# Working with GrIMP Image Products Using nisarImage and nisarImageSeries Classes
---

This notebook illustrates some of the capabilities of the `nisarImage` and `nisarImageSeries` classes for working with GrIMP uncalibrated and calibrated image products. There are two classes, each derived from the same parent class so they have similar functionality. The main difference is that a `nisarImage` instance works with a single image and  date. By constrast, the `nisarImageSeries` can incorporate any number of image products, so long as they have the same geometry (resolution, and extent; e.g., all Greenand NSIDC-0723 image maps of the same type). Calibrated and uncalibrated products can not be combined in the same `nisarImageSeries` since they have different resolutions.

This notebook will help the user to:
- Search and locate Greenland Synthetic Aperture Image (SAR) mosaics covering the period from 2015 to present, including image (scaled for best contrast) and calibrated data sets ($\sigma_o$ and $\gamma_o$).
- Remotely access individual images and time-series of images.
- Create figures and plot results from the data sets.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('Features.md', encoding='utf-8').read()))

## Environment Setup

The following packages are needed to execute this notebook. The notebook has been tested with the `environment.yml` in the *binder* folder of this repository. Thus, for best results, create a new conda environment to run this and other other GrIMP notebooks from this repository. 

`conda env create -f binder/environment.yml`

`conda activate greenlandMapping`

`python -m ipykernel install --user --name=greenlandMapping`

`jupyter lab`

See [NSIDCLoginNotebook](https://github.com/fastice/GrIMPNotebooks/blob/master/NSIDCLoginNotebook.ipynb) for additional information.

The notebooks can be run on a temporary virtial instance (to start click [**binder**](https://mybinder.org/v2/gh/fastice/GrIMPNotebooks/HEAD?urlpath=lab)). See the github [README](https://github.com/fastice/GrIMPNotebooks#readme) for further details.

## Python Setup

In [ ]:
#%matplotlib ipympl
%load_ext autoreload
%autoreload 2
import panel as pn
pn.extension()
import grimpfunc as grimp
import nisardev as nisar
#from matplotlib import colors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import os
import matplotlib.pyplot as plt
import numpy as np
import dask
from dask.diagnostics import ProgressBar
ProgressBar().register()
import matplotlib.gridspec as gridspec


## Help

**Note to get help and see options for any of the GrIMP or other functions while the cursor is positioned inside a method's parentheses, click shift+Tab.**

## Trouble Shooting

NSIDC limits the number of simultaneous connections. As a result, a download can sometimes fail, especially if multiple notebooks are downloading or the ```num_workers``` is set to large. In these cases, try rerunning with only a single notebook downloading or ```num_workers=2``` (current default).

## Login to EarthData/NSIDC

Unless the data have already been downloaded, users will need to sign in to NSIDC/EarthData to run the rest of the notebook. If a ~/.netrc exists, it will load credentials from there. If not, it will create or append to one after the login has been processed since it is needed by GDAL (see [NSIDCLoginNotebook](https://github.com/fastice/GrIMPNotebooks/blob/master/NSIDCLoginNotebook.ipynb)) for more details on security issues.

In [ ]:
# Set path for gdal
env = dict(GDAL_HTTP_COOKIEFILE = os.path.expanduser('~/.grimp_download_cookiejar.txt'),
            GDAL_HTTP_COOKIEJAR = os.path.expanduser('~/.grimp_download_cookiejar.txt'))
os.environ.update(env)
# Get login
myLogin = grimp.NASALogin()  # If login appears not to work, try rerunning this cell
myLogin.view()

## Bounding Box

The examples in this glacier will focus on Zacharie Isstrom in northern Greenland, which can be defined with the following bounding box.

In [ ]:
bbox = {'minx': 440000, 'miny': -1140000, 'maxx': 500000, 'maxy': -1080000}
xbox = np.array([bbox[x] for x in ['minx', 'minx', 'maxx', 'maxx', 'minx']]) * 0.001
ybox = np.array([bbox[y] for y in ['miny', 'maxy', 'maxy', 'miny', 'miny']]) * 0.001

## Search for Data

Greenland Mapping Project data can be searched for using instances of the class, `cmrUrls`, which provides a simple graphical and non-graphical interface to the GMP products. In this example, the search tool is used with `mode=image`, which restricts the search to **NSIDC-0723** image products. The date range can restricted with `firstDate='YYYY-MM-DD'` and `lastDate='YYYY-MM-DD'`. The images are distributed as uncalibrated **image** products or calibrated **sigma0** ($\sigma_o$) and **gamma0** ($\gamma_o$) products, which can be specified as `productFilter='image'`. In the following example, these search will be carried out based on the input parameters, but a gui search window will popup, which allows the search parameters to be altered.

In [ ]:
# For some environments the tool is unresponsive (i.e., search button doesn't work) - this can often be fixed by re-running this cell
myImageUrls = grimp.cmrUrls(mode='image')  # mode image restricts search to the image products
myImageUrls.initialSearch(firstDate='2020-01-01', lastDate='2020-05-01', productFilter='image')

If the search parameters do not need to be altered, then insering a semicolon at the end of the line will supress the output. So the corresponding sigma0 and gamma0 products can searched for as:

In [ ]:
mySigma0Urls = grimp.cmrUrls(mode='image')  # mode image restricts search to the image products
mySigma0Urls.initialSearch(firstDate='2020-01-01', lastDate='2020-05-01', productFilter='sigma0');
myGamma0Urls = grimp.cmrUrls(mode='image')  # mode image restricts search to the image products
myGamma0Urls.initialSearch(firstDate='2020-01-01', lastDate='2020-05-01', productFilter='gamma0');

## To Chunk or Not

There are two ways to read the data, which are controlled by the parameter `useStack`:
1) `useStack=True` (default): In this case, each band of the subsetted data is loaded as a single file read operation (e.g., no chunks in xy). For most cases, this is the faster option because there is no dask overhead and the single reads tend to be faster. This is the preferred option for anything that involves reading in data with `loadRemote` and doing repeated operations on the result. **Note: with this option, the keyword `chunks` will be ignored.**
2) `useStack=False`: In this case, the data are chunked with rio-xarray and dask, which can add 10s of seconds or more overhead to set up the xarray, which slows the lazy reads. There are only a few use cases in which this mode would be preferred. For example, examining only a few points in a large subset **without** using `loadRemote`, so that as the data are input on the fly, only the chunks surrounding the points would be read. In most cases, it's best to minimize the subset area (e.g., for one glacier) or use multiple subsets for widely spaced glaciers.

In [ ]:
useStack=True

## Number of Workers

With Dask you can use `numWorkers` to specify multiple parallel threads, which can speed up downloads. It can also cause the download to fail (with something that looks like a file not found error) if the server decides it's receiving too many concurrent requests. The criteria under which this happens are unclear, and whether it has to do with the number of connections, open files, etc. This means the results could differ for the type of data product/access. For example, downloading a large number of TSX files (many file open operations per unit time) might cause things to break before the case where large pieces are pulled from full ice sheet mosaics (files are held open for long periods). Beyond `numWorkers=8`, the point of diminishing returns is approached. In general, `numWorkers=4` will be fairly robust, and provide good performance. But you can experiment by setting `numWorkers` below.  Note: this discussion is most applicable to network reads. For local file systems, the speedup may be substantially less due to file contention.

**If a download fails, try re-running. If it still fails, try reducing value of `numWorkers`**

In [ ]:
numWorkers = 4

## Loading the data

Now that the data have been located, they can be opened for access. The list of urls is given by `myImageUrls.getCogs()` to be passed into the `readSeriesFromTiff` method.

In [ ]:
%%time
myImageSeries = nisar.nisarImageSeries(numWorkers=numWorkers)  # Instantiate the series object
myImageSeries.readSeriesFromTiff(myImageUrls.getCogs(), useStack=useStack)
myImageSeries.subset  # Display map of data layout - add ; to suppress this output

At more than 100GB, downloading this full data set would take a substantial amount of time, even over a fast network. But if we use the bounding box defined above, the data set can be limited to just the region of interest as follows:

In [ ]:
%%time
myImageSeries.subsetImage(bbox)
myImageSeries.subset

Here the volume as been greatly reduced. At this stage, the data are still on the NSIDC server. At this point several actions can be taken (e.g., displaying the data), which will automatically download the data using dask. While this implicit download is convenient, it can add time for multiple operations on the data. While in principle the data are cached by the OS, they can be flushed from the cache, require re-download. The volume in this example is not large for most computers, so it makse sense to explicily download the data with as shown next:

In [ ]:
%%time
myImageSeries.loadRemote()

The process can now be repeated with the sigma0 and gamma0 products.

In [ ]:
%%time
myGamma0Series = nisar.nisarImageSeries(numWorkers=numWorkers)  # Instantiate the series object
myGamma0Series.readSeriesFromTiff(myGamma0Urls.getCogs(), useStack=useStack)  # Open images with lazy reads, this works well with default chunkSize
myGamma0Series.subsetImage(bbox)  # Clip image area
myGamma0Series.loadRemote()  # Download clipped regions

In [ ]:
%%time
mySigma0Series = nisar.nisarImageSeries(numWorkers=numWorkers)  # Instantiate the series object
mySigma0Series.readSeriesFromTiff(mySigma0Urls.getCogs(), useStack=useStack)  # Open images with lazy reads
mySigma0Series.subsetImage(bbox)  # Clip image area
mySigma0Series.loadRemote()  # Download clipped regions

## Overview Images

The code above download a small subset, but in some cases its nice to have an overview of the full data set. As noted above, a nice feature of COGs is that they include image pyramids. A reduced resolution data set can be created as:

In [ ]:
%%time
myOverviewImage = nisar.nisarImage(numWorkers=numWorkers)  # Instantiate single image object
myOverviewImage.readDataFromTiff(myImageUrls.getCogs()[0],  overviewLevel=4)  # Open image 4->800 m res (2^(n+1) * original res) = 32*.25
myOverviewImage.loadRemote()

## Inspect the Data

An interactive plot to inspect the data can be generated as:

In [ ]:
myImageSeries.inspect()

**If nothing is displayed in the above cell, try re-runing the `inspect()` command.**

## Image Types

As noted above, the GMP Sentinel image mosaics are produced as **image** (byte scaled with colortable stretch to enhance contrast), **sigm0** (calibrated radar cross section), **gamma0** (calibrated cross section that reduces topographic effects). A greater description of the characteristics of these products is beyond the scope of this notebooks but can be found in the [user guide for NSIDC-0723](https://nsidc.org/data/nsidc-0723/). The following cell illustrates how each of these products can be displayed for a given date with the overview image used as an inset map. 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 11))
for mySeries, ax in zip([myImageSeries, mySigma0Series, myGamma0Series], axes):
    mySeries.displayImageForDate(date='2020-02-01', ax=ax, percentile=99)  # Clips to 1% (100-99) and 99 percentile
    ax.axis('off')
height = 3
fig.tight_layout()
axInset = inset_axes(axes[0], width=height * myOverviewImage.sx/myOverviewImage.sy, height=height, loc=2)
myOverviewImage.displayImage(date='2020-02-01', ax=axInset, colorBar=False, axisOff=True,
                             units='km', backgroundColor=(1,1,1,.25),cmap=plt.cm.gray.with_extremes(bad=(.4,0.4,.4)), title='', extend='both')
axInset.plot(xbox, ybox, 'r')

## Image Resolution

The figure above does not capture the full resolution of the data. To better illustrate the 25-m resolution of the data, the following plot zooms in on the center of the image at 4 different levels.

In [ ]:
fig = plt.figure(figsize=(21,12))
# Use gridspec to apportion plot area
m, n = 5, 5
gs = gridspec.GridSpec(2*n, m+2*n)
# Compute center and dimensions of box
xc, yc = (bbox['maxx'] + bbox['minx']) * 0.5 * 0.001, (bbox['maxy'] + bbox['miny']) * 0.5 * 0.001
dx, dy = (bbox['maxx'] - bbox['minx']) * 0.001, (bbox['maxy'] - bbox['miny']) * 0.001
# Display the over view image
axOverview = plt.subplot(gs[:, 0:m])
myOverviewImage.displayImage(ax=axOverview, percentile=99, units='km', colorBarPosition='bottom',
                                     colorBarPad=.25, colorBarSize='2%', midDate=False, masked=False,
                                     backgroundColor=(.9,.9,.9),cmap=plt.cm.gray.with_extremes(bad=(.4,0.4,.4)))
axOverview.axis('off')
# Create axes for zoomed images
axes = [plt.subplot(gs[n*i:n*(i+1), m+n*j:m+n*(j+1)]) for i in range(0, 2) for j in range(0,2)]
# Loop through scale factors
for ax, scale in zip(axes, [1, 2, 4, 8]):
    myImageSeries.displayImageForDate(date='2020-02-01', ax=ax, percentile=99, units='km', title='', colorBarSize='3%')
    # Zoom by adjusting plot area.
    ax.set_xlim((xc-dx*0.5/scale, xc+dx*0.5/scale))
    ax.set_ylim((yc-dy*0.5/scale, yc+dy*0.5/scale))
    # Plot zoom outlines on first image
    axes[0].plot(xc + (xbox-xc)/scale, yc+ (ybox-yc)/scale, color='w')
    # Plot zoom outlines on overview images
    axOverview.plot(xc + (xbox-xc)/scale, yc+ (ybox-yc)/scale, color='r')
fig.tight_layout()

## Statistics

Some basic stats can also be computed for the image series. In the following the mean and standard devation are computed for the stack. Also computed are the anomaly (difference from mean for each time period). In the example below, only the anomly closest to '2020-02-28' is shown. 

In [ ]:
mean = myImageSeries.mean()
anomaly = myImageSeries.anomaly()
sigma = myImageSeries.stdev()
fig, axes = plt.subplots(1, 3, figsize=(24, 11))
for image, ax, vmin, vmax, extraTitle, extend in zip([mean, sigma, anomaly], axes, [0, 0, -10], [160, 20, 10], ['Mean', 'Sigma', 'Anomaly'], ['both', 'max', 'both']):
    # midDate=False -> the title first and last dates in series. No date specified since the stats have a single date
    image.displayImageForDate(date='2020-02-28', ax=ax, vmin=vmin, vmax=vmax, midDate=False, cmap='gray', masked=None, extend=extend)
    # Update existing title
    ax.set_title(f'{extraTitle} for {ax.get_title()}', fontsize=18)
fig.tight_layout()

## Time Series Plots

Values are easily plotted from the image stack. For example, to plot $\sigma_o$ and $\gamma_o$ for the center of the image:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(19, 9))
# Show mean image
mean.displayImageForDate(date='2020-02-28', ax=axes[0], percentile=99, colorBarPosition='top', title='', units='km', colorBarPad=.65)
axes[0].plot(xc, yc, 'r*', markersize=20, ) # plot point
# Plot time series
mySigma0Series.plotPoint(xc, yc, 'r*-', label='$\\sigma_o$', band='sigma0', units='km', ax=axes[1])
myGamma0Series.plotPoint(xc, yc, 'k*-', label='$\\gamma_o$', ax=axes[1], units='km')
# Format plot
myGamma0Series.labelPointPlot(axes[1], title='$\\sigma_o$ and $\\gamma_o$ as function of time',
                              xLabel='Radar Cross Section (dB)', plotFontSize=14)
# Create legend
axes[1].legend(fontsize=16)
# Reduce tick density
xticks = axes[1].get_xticks()
axes[1].set_xticks(xticks[range(0, len(xticks), 2)]); # Reduce tick density
fig.tight_layout()

Now plot all the points in the series along a profile that extends from x in the range of 450 to 490 and y fixed -1110 km.

In [ ]:
x = np.arange(450, 490, .2)
y = np.full(x.shape, -1110)
fig, ax = plt.subplots(1, 1, figsize=(15, 8))
bwr = plt.get_cmap('bwr', len(myGamma0Series.time))
for time, color  in zip(myGamma0Series.time, range(0, len(myGamma0Series.time))):
    # Plot profile for current time value
    myGamma0Series.plotProfile(x, y, '.', date=time, ax=ax, units='km', color=bwr(color), label=time.strftime('%Y-%m-%d'))
# Format plot
myGamma0Series.labelProfilePlot(ax, fontScale=1.3, title='$\\gamma_o$ for x in range from 459 to 490 km with y=-1110 km')
ax.legend(ncol=3, loc='lower left')

## Saving the Data to netCDF

Some downloads could take several minutes (e.g., all 6/12 day maps), in which case it useful to be able to save the data to a single netCDF file for later use. The downloaded subset can be saved in a netcdf and reloaded for to `imageSeries` instance for later analysis. Note if the data have been subsetted, **ONLY** the subset will be saved (\~100MB in this example). If not, the entire Greeland data set will be saved (\~2TB). Before saving, the bounds will be set to the original values, forcing a new download to overwrite the previous.

In [ ]:
myImageSeries.toNetCDF('imageSeries.xyBounds.nc')
# Now reload the data
myImageSeriesReload = nisar.nisarImageSeries()
myImageSeriesReload.readSeriesFromNetCDF('imageSeries.xyBounds.nc')
myImageSeriesReload.loadRemote()
os.remove('imageSeries.xyBounds.nc')  # Comment to keep the file
myImageSeriesReload.subset